# 09G · SPARKLE and SpotClean on matched native windows

Run this notebook beside 09F using the same Python kernel. It selects two disjoint windows within the earlier evaluation core, prepares a single raw count matrix for each window, then fits both methods afresh. The endpoint is how much myeloid signal each method retains in macrophage/monocyte-labeled cells and in neighboring pan-T-supported cells.

The default **128 × 128 µm** window has 4,096 native squares. Its inner **104 × 104 µm** core permits evaluation of complete Proseg footprints. The stored combined `he_tissue_score` supplies staining intensity. No image reader is needed.

In [ ]:
from pathlib import Path
from IPython.display import display, Image, Markdown
import json
import pandas as pd
from bins09d import STUDY_ROOT, TEMP_PROJECT_ROOT
from review09 import show_review_zip
from run09g import run_09g

RESULT_ROOT = Path("09_pilot_results")
RUN_DIR = None         # Optional path to the completed 09B run
RSCRIPT = None         # Reuse the working path in the most recent matching 09F receipt
PATH_OVERRIDES = {}    # Same keys as 09F, including "zarr" and "aligned_qc"
ZARR_TABLE = "square_002um"

N_REGIONS = 2
SIDE_BINS, HALO_BINS = 64, 6
SEARCH_STRIDE_BINS = None  # Default is one eighth of the window width
SCORE_MAX = 0.10
MIN_BACKGROUND_FRACTION = 0.25
REQUIRE_SR_BACKGROUND = False   # True requires low-score unassigned background outside Space Ranger tissue
MIN_MAC_CELLS, MIN_NEAR_T_CELLS = 3, 3
MIN_PANEL_UMIS = 5
SPARKLE_BIN_UM, SPARKLE_RADIUS_UM = 12., 50.
SPOTCLEAN_MINUTES = 60           # Per-region time limit, excluding setup

## Qualify the regions and fit both methods

The search uses prior labels, original polygons and continuous staining scores. It first checks support for two groups: labeled macrophages or monocytes, and non-macrophage cells supported by pan-T markers within 25 µm of a labeled macrophage or monocyte. It also checks coverage of prior T-cell labels. A candidate needs measured raw myeloid signal in **both** primary groups and raw pan-T signal in the nearby group. Cd4 and CD8 counts are recorded independently of selection. The candidate table shows which criteria excluded each window.

Background candidates are unassigned, measured squares with score ≤ `SCORE_MAX`. The selection prioritizes windows with sufficient candidates outside Space Ranger tissue. It also considers candidates within the Space Ranger boundary if permitted. These are provisional background assignments with an explicit model assumption. Missing scores never qualify a square for background.

Each fit reads the same native count file, coordinates, gene plan and cell assignments. SPARKLE fits leakage using out-of-mask counts, whereas SpotClean treats the provisional background as having zero native expression. Their fitted background definitions are recorded separately. A 60-minute limit applies to each SpotClean process. Larger grids can take substantially longer than the 09F grid.

In [ ]:
out, archive, status = run_09g(
    result_root=RESULT_ROOT, run_dir=RUN_DIR, study_root=STUDY_ROOT, temp_project_root=TEMP_PROJECT_ROOT,
    overrides=PATH_OVERRIDES, zarr_table=ZARR_TABLE, rscript=RSCRIPT, n_regions=N_REGIONS,
    side_bins=SIDE_BINS, halo_bins=HALO_BINS, stride_bins=SEARCH_STRIDE_BINS,
    score_max=SCORE_MAX, min_background_fraction=MIN_BACKGROUND_FRACTION,
    require_sr_background=REQUIRE_SR_BACKGROUND, min_mac_cells=MIN_MAC_CELLS,
    min_near_t_cells=MIN_NEAR_T_CELLS, min_panel_umis=MIN_PANEL_UMIS,
    sparkle_bin_um=SPARKLE_BIN_UM, sparkle_radius_um=SPARKLE_RADIUS_UM, fit_minutes=SPOTCLEAN_MINUTES)
print("Selected regions:", status["selected"], " |  Verified pairs:", status["verified_pairs"])
display(pd.DataFrame(status["stages"]).T.fillna(""))
review_download = show_review_zip(archive)

## Tissue and endpoint coverage

The maps show the original continuous score and SpotClean role for each selected square. The candidate counts explain a shortfall when fewer windows qualify. Counts for low-abundance coreceptors may still be zero in an otherwise informative window.

In [ ]:
regions = out / "regions"
if (regions / "selection.json").is_file():
    selection = json.loads((regions / "selection.json").read_text())
    display(pd.Series(selection.get("decision_counts", {}), name="candidate windows"))
    if selection.get("shortfall"):
        print(selection["shortfall"])
if (regions / "candidates.csv").is_file():
    audit = pd.read_csv(regions / "candidates.csv")
    cols = ["region","row","col","background_mode","n_complete","n_mac","n_near_t","n_t_labels",
            "myeloid_in_mac","myeloid_in_near_t","panT_in_near_t","Cd4_in_T_labels","CD8_in_T_labels","decision"]
    if len(audit):
        cols = [c for c in cols if c in audit]
        show = audit[audit.decision.eq("selected")]
        if show.empty:
            sort = [c for c in ("n_near_t","n_mac","n_bg_either_sr") if c in audit]
            show = (audit.sort_values(sort, ascending=False) if sort else audit).head(12)
        display(show[cols])
for p in sorted(regions.glob("region_*/score_and_roles.png")):
    print(p.parent.name)
    display(Image(filename=str(p), width=780))

## Paired results

Read the source and neighbor rows together. For the same decrease in myeloid counts in nearby T-supported cells, greater retention in labeled macrophages is preferable. The raw UMI denominator, number of cells and gene-specific fit status distinguish a numerical change from a meaningful comparison. A SpotClean retention above 100% is possible. An unchanged SPARKLE gene may have failed its leakage-fit threshold.

The pooled table sums raw and corrected counts across verified, nonoverlapping windows. Each window comes from the same specimen and was chosen for its immune-cell coverage. The result is a local stress test. Sources outside a crop can affect cells near its edge.

In [ ]:
comparison = out / "comparison"
if (comparison / "paired_readout.md").is_file():
    display(Markdown((comparison / "paired_readout.md").read_text()))
if (comparison / "paired_retention.png").is_file():
    display(Image(filename=str(comparison / "paired_retention.png"), width=950))
for name in ("pooled_retention.csv", "region_status.csv"):
    path = comparison / name
    if path.is_file():
        display(pd.read_csv(path).round(3))
fits = []
for p in sorted((out / "models").glob("region_*/*/fit.json")):
    row = json.loads(p.read_text())
    fits.append({"region":p.parent.parent.name, "method":row["method"],
                 "seconds":row.get("wall_seconds"), "R peak MiB":row.get("peak_R_RSS_MiB"),
                 "spatial genes":row.get("n_spatial_genes"), "SPARKLE lambda (µm)":row.get("lambda_um"),
                 "SPARKLE target genes passing":row.get("targets_passing_gate"),
                 "SpotClean distal rate":row.get("distal_rate"), "SpotClean radius (µm)":row.get("radius_um")})
if fits:
    display(pd.DataFrame(fits).round(3).fillna(""))
print("Return ZIP", review_download)